# Figure 4: Exploratory Data Analysis and Descriptive Statistics Visualization

## Purpose
Generate **descriptive visualization** of IPC phase distributions across space and time, showing:
- Global geographic distribution of food crisis observations by country
- Temporal trends in phase classifications from 2017-2022
- Phase composition changes over years

## Data contract
Run this notebook from `2.Source Code/`. It uses the released `../1.Source Data/Nowcasting_Analysis_010825.csv` table and joins `../1.Source Data/area_country_lookup.csv` on `area_id`; the archived 252 MiB pre-merge table is not required. Outputs are written to `produced_graph/`.

## Dependencies
- **geopandas**: Country boundary visualization
- **pandas, numpy**: Data manipulation
- **matplotlib**: Visualization
- GeoPandas' bundled Natural Earth low-resolution country boundaries

Figures are descriptive and do not contain model predictions.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

OUTPUT_DIR = Path('produced_graph')
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
analysis = pd.read_csv('../1.Source Data/Nowcasting_Analysis_010825.csv')
area_country = pd.read_csv('../1.Source Data/area_country_lookup.csv')
data = analysis.merge(area_country, on='area_id', how='left', validate='many_to_one')
if data['country_code_3'].isna().any():
    raise ValueError('area_country_lookup.csv does not cover every modeled area_id')
country_counts = data['country_code_3'].value_counts().rename_axis('country_code_3').reset_index(name='count')

# Load the world map from Natural Earth dataset
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# Merge the data with the world map
world = world.merge(country_counts, how='left', left_on='iso_a3', right_on='country_code_3')

# Create a custom colormap using a subset of the Blues colormap
cmap = mcolors.LinearSegmentedColormap.from_list(
    'custom_blues', 
    plt.cm.Blues(np.linspace(0.4, 1, 256))
)

# Plot the map with adm0 level basemap
fig, ax = plt.subplots(1, 1, figsize=(15, 10))
world.boundary.plot(ax=ax, linewidth=1, edgecolor='black')  # Plot the adm0 level boundaries
world.plot(column='count', ax=ax, legend=True, cmap=cmap, 
           legend_kwds={'label': "Frequency by Country",
                        'orientation': "horizontal"})

# Add title and remove axis for better visual
ax.set_title('World Map with Visits Frequency by Country', fontdict={'fontsize': 20}, pad=20)
ax.set_axis_off()
plt.savefig(
    OUTPUT_DIR / 'Picture10.jpg', 
    dpi=300,
    format='jpeg',
    bbox_inches='tight',
    pil_kwargs={'compression': 'lzw'}
)

plt.show()


In [ ]:
# import data
data = pd.read_csv(r'../1.Source Data/Nowcasting_Analysis_010825.csv')

In [ ]:


# Count the frequency of each value
phase_counts = data['overall_phase'].value_counts().sort_index()

# Define custom colors similar to the image
colors =[ '#009E73','#56B4E9','#0072B2','#E69F00', '#D55E00'] 

# Create the plot
plt.figure(figsize=(10, 6))
bars = plt.bar(phase_counts.index, phase_counts.values, color=colors)
ax = plt.gca()

# Add frequency labels just above each bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 5,  # small offset for visibility
             f'{int(height)}',
             ha='center', va='bottom', fontsize=10)

# Add labels and title
plt.xlabel('overall phase', fontsize=12)
plt.ylabel('count', fontsize=12)
#put the title under the x-axis
plt.text(0.75, -0.62, '(A) Count of Overall IPC Phases',
         fontsize=12, fontweight='bold', ha='center', va='bottom',transform=ax.transAxes)

# Set the x-ticks to be integer values 1-5
plt.xticks(np.arange(1, 6))

# Add grid lines for better readability (only horizontal)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add a thin border
plt.box(True)

# Adjust ylim to make room for the frequency labels
plt.ylim(0, phase_counts.max() * 1.1)  # Add 10% padding at the top

# Maximize space for the plot
plt.tight_layout()

# Save the figure (optional)
#plt.savefig('overall_phase_distribution.png', dpi=300)
plt.savefig(
    OUTPUT_DIR / 'Picture1.jpg', 
    dpi=300,
    format='jpeg',
    bbox_inches='tight',
    pil_kwargs={'compression': 'lzw'}
)


# Show the plot
plt.show()

In [ ]:
# convert date to datetime and derive the plotting year
data['date'] = pd.to_datetime(data['date'])
data['year'] = data['date'].dt.year

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Create dataframe
df = pd.DataFrame(data)

# Calculate percentages of each phase by year
phase_counts = df.groupby(['year', 'overall_phase']).size().unstack(fill_value=0)

# Calculate the percentage for each phase within each year
phase_percentages = phase_counts.div(phase_counts.sum(axis=1), axis=0) * 100

# Define the colors for phases 2, 3, and 4
colors =[ '#009E73','#56B4E9','#0072B2','#E69F00', '#D55E00'] # Blue, Orange, Green

# Create the stacked bar chart
ax = phase_percentages.plot(kind='bar', stacked=True, figsize=(10, 6), 
                           color=colors, width=0.7)

# Customize plot
plt.xlabel('Year', fontsize=12)
plt.ylabel('Percentage', fontsize=12)
plt.ylim(0, 100)
plt.xticks(rotation=0)  # Horizontal x-axis labels
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend(title='Overall Phase', loc='upper right')
plt.text(0.5, -0.2, '(B) Percentage of Overall IPC Phases by Year',
         fontsize=12, fontweight='bold', ha='center', va='bottom',transform=ax.transAxes)

# Remove the border on the right and top
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

# Tight layout
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'Picture2.jpg', 
    dpi=300,
    format='jpeg',
    bbox_inches='tight',
    pil_kwargs={'compression': 'lzw'}
)

# Save the figure
#plt.savefig('phase_percentage_by_year.png', dpi=300)

# Show the plot
plt.show()